# 05 — Q3: Markets

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.db import get_conn, register_csv_tables, materialize_sql_file
from src.markets import per_card_profit_with_ci
from src.wise_theme import (
    apply as apply_wise_theme,
    BRIGHT_GREEN, DARK_GREEN, MID_GREEN, SOFT_GREEN, LIGHT_BG, WARM_GREY,
    diverging_cmap, sequential_cmap, style_axes, value_color, label_color_on,
)

FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
apply_wise_theme()

con = get_conn()
register_csv_tables(con)
for f in ["01_clean_transactions","02_cost_lookup","03_profit_per_transaction",
          "04_card_cohort"]:
    materialize_sql_file(con, ROOT / "sql" / f"{f}.sql")
scorecard = materialize_sql_file(con, ROOT / "sql" / "06_market_scorecard.sql")
print(f"{len(scorecard)} markets in dataset")

## 1. Market scorecard

In [ ]:
top = scorecard[scorecard["n_cards"] >= 60].copy()
display_cols = ["market","n_cards","pct_active","tx_per_active_card","avg_billing_gbp",
                "avg_profit_per_tx_gbp","observed_profit_per_card_gbp",
                "pct_inter","pct_intra","pct_domestic","pct_atm","fail_pct"]
top[display_cols].round(3)

## 2. Confidence in per-card profit comparisons

In [ ]:
# Per-card observed profit (sum profit_gbp by card, joined to market)
per_card = con.execute('''
    SELECT cc.profile_address_country AS market,
           ppt.card_token,
           SUM(ppt.profit_gbp) AS profit_gbp
    FROM profit_per_transaction ppt
    JOIN card_cohort cc USING (card_token)
    GROUP BY cc.profile_address_country, ppt.card_token
''').df()

# Cards with no SUCCESS tx must be added back as zero-profit so means reflect the full
# denominator (otherwise we're averaging only over active cards).
all_cards = con.execute('''
    SELECT cc.profile_address_country AS market, cc.card_token
    FROM card_cohort cc
''').df()
per_card_full = (all_cards
    .merge(per_card, on=["market","card_token"], how="left")
    .fillna({"profit_gbp": 0.0}))

ci = per_card_profit_with_ci(per_card_full, group_col="market", min_n=30)
ci

In [ ]:
per_card = con.execute('''
    SELECT cc.profile_address_country AS market, ppt.card_token,
           SUM(ppt.profit_gbp) AS profit_gbp
    FROM profit_per_transaction ppt
    JOIN card_cohort cc USING (card_token)
    GROUP BY cc.profile_address_country, ppt.card_token
''').df()
all_cards = con.execute('''
    SELECT cc.profile_address_country AS market, cc.card_token FROM card_cohort cc
''').df()
per_card_full = (all_cards.merge(per_card, on=["market","card_token"], how="left")
                          .fillna({"profit_gbp": 0.0}))
ci = per_card_profit_with_ci(per_card_full, group_col="market", min_n=30)
ci = ci.sort_values("mean_profit_per_card", ascending=True).reset_index(drop=True)

point_colors = [value_color(v) for v in ci["mean_profit_per_card"]]

fig, ax = plt.subplots(figsize=(9.5, 0.36*len(ci) + 1.8))
y = np.arange(len(ci))
for i, (mean, lo, hi, c) in enumerate(zip(ci["mean_profit_per_card"],
                                          ci["ci_lo"], ci["ci_hi"], point_colors)):
    ax.plot([lo, hi], [i, i], color=DARK_GREEN, linewidth=1.6, solid_capstyle="round", zorder=3)
    ax.plot(mean, i, "o", markersize=10, color=c,
            markeredgecolor=DARK_GREEN, markeredgewidth=1.3, zorder=4)
ax.axvline(0, color=DARK_GREEN, lw=0.8, ls=(0, (4, 3)), alpha=0.5)
ax.set_yticks(y); ax.set_yticklabels(ci["market"], fontweight="bold")

xmax_data = ci["ci_hi"].max()
xmin_data = ci["ci_lo"].min()
xpad = (xmax_data - xmin_data) * 0.16
ax.set_xlim(xmin_data - 0.2, xmax_data + xpad)
n_label_x = xmax_data + xpad * 0.25
for i, n in enumerate(ci["n_cards"]):
    ax.text(n_label_x, i, f"n={int(n)}",
            va="center", ha="left", fontsize=9, color=WARM_GREY, fontweight="bold")
ax.set_xlabel("Observed 4-month profit per card (£)")
ax.set_title("Per-market profit/card  •  mean ± 95% bootstrap CI")
ax.tick_params(axis="x", length=0); ax.tick_params(axis="y", length=0)
fig.tight_layout()
fig.savefig(FIG / "fig5_q3_market_ci.png")
plt.show()

## Appendix — Decline reasons

In [ ]:
def classify(reason):
    if reason in ("INSUFFICIENT_FUNDS","INCORRECT_PIN","INCORRECT_CVV",
                  "PIN_ENTRY_TRIES_EXCEEDED","CARD_INACTIVE"):
        return "Customer (insufficient funds / wrong PIN / not yet activated)"
    if reason in ("PAYMENT_METHOD_TRANSACTION_LIMIT_EXCEEDED",
                  "PAYMENT_METHOD_DAILY_LIMIT_EXCEEDED",
                  "PAYMENT_METHOD_MONTHLY_LIMIT_EXCEEDED",
                  "CARD_FROZEN"):
        return "Ambiguous (user-set or Wise-default)"
    return "System / other"

all_decline = con.execute(
    "SELECT decline_reason, COUNT(*) AS n FROM clean_transactions "
    "WHERE state='FAIL' GROUP BY decline_reason ORDER BY n DESC"
).df()
all_decline["category"] = all_decline["decline_reason"].map(classify)
all_decline

In [ ]:
category_summary = (all_decline.groupby("category", as_index=False).agg(n=("n","sum"))
                    .sort_values("n", ascending=False))
category_summary["pct"] = (category_summary["n"] / category_summary["n"].sum() * 100).round(1)
category_summary

In [ ]:
# Persist for deck/figures
RESULTS = ROOT / "data" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
scorecard.to_csv(RESULTS / "market_scorecard.csv", index=False)
ci.to_csv(RESULTS / "market_profit_per_card_ci.csv", index=False)
all_decline.to_csv(RESULTS / "decline_reasons.csv", index=False)
print("Persisted Q3 outputs to data/results/")